# EOA Framework — 01. Offline / Mock Walkthrough (Zero External Tokens)

This notebook demonstrates the complete **ExplanationLayer** developer experience (DX) running **100% deterministically and offline** using simulated industrial telemetry (UCI Hydraulic Test Rig) and a custom local callable/MockLLM.

**Scenarios covered:**
1. **Plug & Play (2 Lines)**: `ExplanationLayer.default_layer` + `layer.explain(...)` in Direct (Push) Mode.
2. **"Developer is King"**: Customizing agents (`remove_agent`, `override_agent_prompt`, `SpecializedAgent`, `inspect`).
3. **Multi-Turn Dialogue & SQLite Audit**: Conversational follow-ups with automatic artifact caching and SQL inspection.
4. **Autonomous Query Mode (Pull)**: Binding Python tools (`register_tool`) and triggering explanations via `incident_id`.
5. **"Unplugged" Fail-Safe Mode**: Proving the host system never crashes when the LLM fails.

## 1. Setup & Simulated Telemetry Testbed

In [ ]:
import sys
from pathlib import Path

# Ensure project root/src is in Python path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR / "src"))
sys.path.insert(0, str(ROOT_DIR))

import eoa
from eoa import (
    ExplanationLayer,
    SpecializedAgent,
    SQLiteSessionStore,
)
from tests.mocks.mock_llm import MockLLM

print(f"EOA Framework version: {eoa.__version__}")

In [ ]:
# Simulated SCADA / MLflow Incident Database
INCIDENTS_DATABASE = {
    402: {
        "prediction": {
            "model_name": "XGBoost_CoolerCondition_v2",
            "target_variable": "Cooler Condition Tier",
            "predicted_value": "Reduced Efficiency (Tier 3)",
            "confidence_or_probability": 0.86,
            "status_label": "Degraded Operation / Critical Warning",
        },
        "xai": {
            "method": "SHAP",
            "feature_attributions": {
                "PS1_Inlet_Pressure": 0.54,
                "EPS1_Motor_Power": 0.32,
                "TS2_HeatExchanger_Temp": 0.14,
            },
        },
        "context": {
            "variables": {
                "PS1_Inlet_Pressure": "Bar, normal operating threshold [2.0, 4.0]",
                "EPS1_Motor_Power": "kW, normal operating threshold [1.5, 2.8]",
            },
            "thresholds": {"PS1_max": "4.0 bar", "EPS1_max": "3.0 kW"},
            "documents": {
                "SOP_Cooler_04": "Section 4: If PS1 exceeds 4.0 bar accompanied by EPS1 > 3.0 kW, cavitation in the suction filter is probable. Reduce hydraulic load immediately."
            },
        },
    }
}

def fetch_scada_incident_data(incident_id: int) -> dict:
    """Autonomous tool matching `incident_id` parameter."""
    print(f"-> [Tool Executed] Retrieved artifacts for incident_id={incident_id}")
    return INCIDENTS_DATABASE.get(incident_id, {})

# Deterministic MockLLM (simulates CMA, AA, EA, VA without spending API tokens)
def _deterministic_responder(prompt: str, system_prompt: str = None) -> str:
    if "Validator" in prompt:
        return '{"score": 9.4, "issues": [], "critique": ""}'
    if "Context Manager" in prompt:
        return "Key Variables:\n- PS1_Inlet_Pressure: [2.0, 4.0] bar.\nOperational Rules: SOP_Cooler_04."
    if "Artifacts Analyst" in prompt:
        return "Outcome: Reduced Efficiency (86% risk). Top Driver: PS1_Inlet_Pressure (+0.54)."
    if "Explainer" in prompt:
        return (
            "O sistema detectou degradação de eficiência no circuito de resfriamento (risco de 86%).\n\n"
            "A causa raiz primária é a sobrepressão no sensor PS1 (+0.54 SHAP), que excedeu 4.0 bar em conjunto com sobrepotência elétrica no motor EPS1.\n\n"
            "Recomendação: Aplicar imediatamente o procedimento SOP_Cooler_04 reduzindo a carga do circuito e inspecionando o filtro de sucção."
        )
    return "Parecer técnico do especialista concluído."

mock_llm = MockLLM(custom_generator=_deterministic_responder)
print("Offline testbed ready!")

## 2. Scenario 1: Plug & Play in 2 Lines (`ExplanationLayer.default_layer`)

In [ ]:
layer = ExplanationLayer.default_layer(llm=mock_llm)

response = layer.explain(
    prediction=INCIDENTS_DATABASE[402]["prediction"],
    xai=INCIDENTS_DATABASE[402]["xai"],
    context=INCIDENTS_DATABASE[402]["context"],
    user_query="Explique o alerta e indique o que verificar.",
)

print("=== NARRATIVA GERADA ===")
print(response.narrative)
print(f"\nQuality Gate Score: {response.validation_score}/10.0 (Passed: {response.validation_passed})")

## 3. Scenario 2: "Developer is King" (Customizing Agents & Prompt Overrides)

In [ ]:
custom_layer = ExplanationLayer(llm=mock_llm)
custom_layer.create_default_agents()

# 1. Override Explainer prompt:
custom_layer.override_agent_prompt(
    "explainer",
    "Você é um engenheiro mecânico sênior. Responda em tom direto: {user_query}",
)

# 2. Register a SpecializedAgent:
custom_layer.register_agent(
    SpecializedAgent(
        name="cavitation_specialist",
        role="Especialista em Cavitação de Válvulas",
        prompt="Avalie os índices de cavitação para: {user_query}",
    )
)

# 3. Inspect layer configuration:
custom_layer.inspect();

## 4. Scenario 3: Multi-Turn Conversation & Local SQLite Audit

In [ ]:
db_path = "mock_audit_sessions.db"
store = SQLiteSessionStore(db_path=db_path)
chat_layer = ExplanationLayer.default_layer(llm=mock_llm, session_store=store)
session_id = "incidente_usina_402"

# Turn 1: Alerta inicial
res1 = chat_layer.explain(
    prediction=INCIDENTS_DATABASE[402]["prediction"],
    xai=INCIDENTS_DATABASE[402]["xai"],
    context=INCIDENTS_DATABASE[402]["context"],
    user_query="O que causou o alerta de sobrepressão?",
    session_id=session_id,
)

# Turn 2: Pergunta de acompanhamento (herda prediction/xai/context da sessão automaticamente!)
res2 = chat_layer.explain(
    user_query="Há risco de rompimento se mantermos a vazão atual por mais 30 minutos?",
    session_id=session_id,
)

print(f"Mensagens persistidas no SQLite ('{db_path}'): {len(store.get_history(session_id))}")
print("Logs de Validação (Quality Gate):", store.get_validation_logs(session_id))

## 5. Scenario 4: Autonomous Query Mode (Pull) & "Unplugged" Fail-Safe

In [ ]:
# Autonomous Pull Mode via Tool
pull_layer = ExplanationLayer.default_layer(llm=mock_llm)
pull_layer.get_agent("artifacts").register_tool(fetch_scada_incident_data)

res_pull = pull_layer.explain(incident_id=402, user_query="Diagnostique o incidente 402")
print("Pull Mode Validated:", res_pull.validation_passed)

# Unplugged Fail-Safe Mode (Simulating total LLM outage)
def crashing_llm(prompt: str) -> str:
    raise ConnectionError("503 Service Unavailable")

failsafe_layer = ExplanationLayer.default_layer(llm=crashing_llm)
res_fail = failsafe_layer.explain(
    prediction=INCIDENTS_DATABASE[402]["prediction"],
    xai=INCIDENTS_DATABASE[402]["xai"],
    user_query="Status check",
)
print("\nFail-Safe Triggered (`fallback_used`):", res_fail.fallback_used)
print(res_fail.narrative)